# 03 — Scaling Analysis (ChestX-ray14)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
from pathlib import Path
import json
import glob
import os

sns.set_theme(style='whitegrid')

In [ ]:
BENCHMARK_DIR = ''  # directory containing all benchmark CSV files
FIGURES_DIR = Path(BENCHMARK_DIR) / 'figures' if BENCHMARK_DIR else Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
BASE_SIZES = [10_000, 50_000, None]  # None = full
BASE_LABELS = ['10k', '50k', 'Full']

In [ ]:
def load_and_merge(prefix):
    pattern = f'{BENCHMARK_DIR}/{prefix}_seed*.csv' if BENCHMARK_DIR else f'{prefix}_seed*.csv'
    files = glob.glob(pattern)
    if not files:
        return pd.DataFrame()
    
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        if 'seed' not in df.columns:
            seed_str = Path(f).stem.split('_seed')[-1]
            if seed_str.isdigit():
                df['seed'] = int(seed_str)
        dfs.append(df)
        
    combined = pd.concat(dfs, ignore_index=True)
    if 'DATASET' in combined.columns:
        combined = combined[combined['DATASET'] == 'chest'].copy()
    elif 'dataset' in combined.columns:
        combined = combined[combined['dataset'] == 'chest'].copy()
    return combined

df_qual = load_and_merge('benchmark_quality')
df_time = load_and_merge('benchmark_timing')
df_cand = load_and_merge('benchmark_candidates')

print("Quality shapes:", df_qual.shape)
print("Timing shapes:", df_time.shape)
print("Candidates shapes:", df_cand.shape)

if not df_qual.empty:
    print("Available seeds (quality):", df_qual['seed'].unique() if 'seed' in df_qual.columns else 'N/A')

In [ ]:
# Merge dataframes
if not df_qual.empty and not df_time.empty and not df_cand.empty:
    merge_keys = ['dataset', 'base_size', 'method', 'seed']
    if 'DATASET' in df_qual.columns: merge_keys[0] = 'DATASET'
    merge_keys = [k for k in merge_keys if k in df_qual.columns]
    
    df_merged = df_qual.merge(df_time, on=merge_keys, how='inner')\
                       .merge(df_cand, on=merge_keys, how='inner')
    
    df_merged['base_size'] = df_merged['base_size'].fillna(-1).replace('None', -1).astype(float)
    
    agg_cols = ['recall@10', 'median_latency_s', 'mean_cand_fraction', 'build_time_s']
    agg_cols = [c for c in agg_cols if c in df_merged.columns]
    
    agg_df = df_merged.groupby(['base_size', 'method'])[agg_cols].agg(['mean', 'std']).reset_index()
    
    display(agg_df)
else:
    print("Dataframes are empty, cannot merge.")
    df_merged = pd.DataFrame()

In [ ]:
if not df_merged.empty:
    plt.figure(figsize=(8, 6))
    
    methods = df_merged['method'].unique()
    colors = {'linear': 'blue', 'SimHash': 'orange', 'E2LSH': 'green'}
    
    for m in methods:
        m_data = df_merged[df_merged['method'] == m]
        m_agg = m_data.groupby('base_size')['median_latency_s'].agg(['mean', 'std']).reset_index()
        x_vals = m_agg['base_size'].replace(-1.0, 112120)  # Approx full size for ChestX-ray14
        
        plt.errorbar(x_vals, m_agg['mean'], yerr=m_agg['std'], label=m, 
                     marker='o', capsize=5, color=colors.get(m, None))

    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Base Size')
    plt.ylabel('Median Latency (s)')
    plt.title('Query Latency vs. Base Size — ChestX-ray14')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig07_scaling_latency.pdf')
    plt.savefig(FIGURES_DIR / 'fig07_scaling_latency.png')
    plt.show()

In [ ]:
if not df_merged.empty and 'recall@10' in df_merged.columns:
    plt.figure(figsize=(8, 6))
    
    for m in methods:
        m_data = df_merged[df_merged['method'] == m]
        m_agg = m_data.groupby('base_size')['recall@10'].agg(['mean', 'std']).reset_index()
        x_vals = m_agg['base_size'].replace(-1.0, 112120)
        
        plt.errorbar(x_vals, m_agg['mean'], yerr=m_agg['std'], label=m, 
                     marker='s', capsize=5, color=colors.get(m, None))

    plt.xscale('log')
    plt.xlabel('Base Size')
    plt.ylabel('Recall@10')
    plt.title('Recall vs. Base Size — ChestX-ray14')
    plt.ylim(0.0, 1.05)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig07_scaling_recall.png')
    plt.show()

In [ ]:
if not df_merged.empty and 'mean_cand_fraction' in df_merged.columns:
    plt.figure(figsize=(8, 6))
    
    for m in methods:
        if m == 'linear': continue
        m_data = df_merged[df_merged['method'] == m]
        m_agg = m_data.groupby('base_size')['mean_cand_fraction'].agg(['mean', 'std']).reset_index()
        x_vals = m_agg['base_size'].replace(-1.0, 112120)
        
        plt.errorbar(x_vals, m_agg['mean'], yerr=m_agg['std'], label=m, 
                     marker='^', capsize=5, color=colors.get(m, None))

    plt.xscale('log')
    plt.xlabel('Base Size')
    plt.ylabel('Candidate Fraction')
    plt.title('Candidate Fraction vs. Base Size — ChestX-ray14')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig03_candidate_fraction.png')
    plt.show()

In [ ]:
if not df_merged.empty and 'build_time_s' in df_merged.columns:
    plt.figure(figsize=(10, 6))
    
    sns.barplot(data=df_merged, x='base_size', y='build_time_s', hue='method')
    
    plt.xlabel('Base Size (-1.0 = Full)')
    plt.ylabel('Build Time (s)')
    plt.title('Build Time by Method and Base Size — ChestX-ray14')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig04_build_time.png')
    plt.show()

In [ ]:
if not df_merged.empty:
    mean_latencies = df_merged.groupby(['base_size', 'method'])['median_latency_s'].mean().unstack()
    
    if 'linear' in mean_latencies.columns:
        speedup_df = pd.DataFrame(index=mean_latencies.index)
        for m in mean_latencies.columns:
            if m != 'linear':
                speedup_df[f'{m}_speedup'] = mean_latencies['linear'] / mean_latencies[m]
        
        print("Speedup relative to linear scan:")
        display(speedup_df)
    else:
        print("Linear method not found, cannot compute speedup.")

### Scaling Summary

1. **Latency Scaling**: E2LSH and SimHash typically show sub-linear scaling, meaning their relative speedup over an exact linear scan increases as the dataset grows.
2. **Recall Stability**: Recall should remain robust across dataset sizes, typically very close to 1.0 for linear and roughly constant for well-calibrated LSH.
3. **Candidate Fraction**: As the base size increases, the candidate fraction often decreases or stays flat for LSH, maintaining its filtering efficiency.
4. **Build Time**: The build time for LSH structures increases linearly or $O(N \log N)$ with the base size, which represents the upfront indexing cost.